# Dacon WPT-XLSR-AASIST 제출 패키지 생성기

이 노트북은 Dacon 공개 Baseline의 **PANNs → HTDemucs → 성분별 Fake 탐지 → MAX Fusion** 구조를 유지하면서, `DF-Arena 1B`만 논문의 **WPT-XLSR-AASIST**로 교체합니다.

```text
INPUT AUDIO
├─ PANNs Cnn14 ────────────────> VOICE/MUSIC_PRESENT_PROB
└─ HTDemucs
   ├─ vocals ───────> WPT-XLSR-AASIST ─> VOICE_FAKE_PROB
   └─ accompaniment -> WPT-XLSR-AASIST ─> MUSIC_FAKE_PROB

max(VP × VF, MP × MF) ─────────> FILE_FAKE_PROB
```

> **중요:** 공식 `xieyuankun/All-Type-ADD` 저장소는 2026-08-31 기준 모델 코드와 결과 파일만 제공하고, 실제 사전학습 체크포인트 `anti-spoofing_feat_model.pt`는 제공하지 않습니다. 따라서 아래 설정의 `WPT_CHECKPOINT` 또는 `WPT_CHECKPOINT_URL`에 유효한 체크포인트를 지정해야 합니다. 체크포인트 없이 임의 가중치로 `submit.zip`을 만들지 않도록 의도적으로 중단 장치를 두었습니다.

공식 출처: [WPT 논문](https://doi.org/10.1609/aaai.v40i42.40907), [공식 코드](https://github.com/xieyuankun/All-Type-ADD), [XLS-R 300M](https://huggingface.co/facebook/wav2vec2-xls-r-300m), [Dacon 대회](https://dacon.io/competitions/official/236749/overview/description)


## 제출 규격 반영 사항

- zip 최상위: `model/`, `script.py`, `requirements.txt`만 포함
- 입력: 평가 서버가 추가하는 `data/test/`, `data/sample_submission.csv`
- 출력: `output/submission.csv`
- 다섯 확률 열 및 sample submission ID 순서 보존
- 인터넷이 차단된 추론 환경을 위해 PANNs·HTDemucs·XLS-R·WPT 체크포인트를 모두 zip 안에 포함
- Dacon 제한(압축 10GB, 해제 32GB) 자동 검사
- 평가 데이터의 파일 단위 독립 예측 원칙 유지


In [12]:
from pathlib import Path
import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
import urllib.request
import zipfile

# ── 사용자가 지정할 값 ───────────────────────────────────────────────
# Dacon 데이터 탭에서 받은 open.zip 또는 그 안의 baseline_submit.zip
OPEN_OR_BASELINE_ZIP = Path.home() / 'Downloads' / 'open.zip'

# 공식 WPT-XLSR-AASIST 학습 체크포인트. 둘 중 하나만 지정하세요.
WPT_CHECKPOINT = Path(r'C:\Users\Admin\Desktop\Dacon-Contest-DL\WPT_XLSR_AASIST_FoR_20000\best.pt')  # 예: Path('/path/to/anti-spoofing_feat_model.pt')
WPT_CHECKPOINT_URL = ''    # 직접 다운로드 URL이 있을 때만 사용

# 생성 위치
WORK_DIR = Path.cwd() / 'wpt_submit_work_2'
OUTPUT_DIR = Path.cwd() / 'output_artifacts'
SUBMIT_ZIP = OUTPUT_DIR / 'submit.zip'

# 공식 코드 버전과 기반 모델
WPT_REPO = 'https://github.com/xieyuankun/All-Type-ADD.git'
WPT_COMMIT = '4faae8ba700aa93f03cfc87c100dc626b6a7e68b'
XLSR_REPO = 'facebook/wav2vec2-xls-r-300m'

RUN_LOCAL_SMOKE_TEST = False  # 유효 체크포인트와 CUDA 환경에서만 True 권장
print('설정 완료:', WORK_DIR.resolve())


설정 완료: C:\Users\Admin\Desktop\Dacon-Contest-DL\wpt_submit_work_2


In [13]:
def run(command, cwd=None):
    print('+', ' '.join(map(str, command)))
    subprocess.run(command, cwd=cwd, check=True)

def sha256(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as f:
        while chunk := f.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()

def safe_extract(zip_path, destination):
    destination = Path(destination).resolve()
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as zf:
        for member in zf.infolist():
            target = (destination / member.filename).resolve()
            if destination not in target.parents and target != destination:
                raise ValueError(f'Unsafe zip member: {member.filename}')
        zf.extractall(destination)

def find_baseline_submit(source_zip, extract_root):
    source_zip = Path(source_zip).expanduser().resolve()
    if not source_zip.is_file():
        raise FileNotFoundError(
            f'{source_zip} 파일이 없습니다. Dacon 데이터 탭에서 open.zip을 받은 뒤 '
            'OPEN_OR_BASELINE_ZIP을 수정하세요.'
        )
    if source_zip.name == 'baseline_submit.zip':
        return source_zip, None

    safe_extract(source_zip, extract_root)
    matches = list(extract_root.rglob('baseline_submit.zip'))
    if len(matches) != 1:
        raise FileNotFoundError(f'baseline_submit.zip을 하나만 찾아야 합니다: {matches}')
    data_roots = [p.parent for p in extract_root.rglob('sample_submission.csv')]
    data_root = data_roots[0].parent if data_roots else None
    return matches[0], data_root

def resolve_checkpoint(cache_dir):
    configured = str(WPT_CHECKPOINT).strip()
    if configured and configured != '.':
        checkpoint = Path(configured).expanduser().resolve()
        if checkpoint.is_file():
            return checkpoint
    if WPT_CHECKPOINT_URL.strip():
        checkpoint = cache_dir / 'anti-spoofing_feat_model.pt'
        checkpoint.parent.mkdir(parents=True, exist_ok=True)
        if not checkpoint.is_file():
            print('WPT 체크포인트 다운로드 중...')
            urllib.request.urlretrieve(WPT_CHECKPOINT_URL, checkpoint)
        return checkpoint.resolve()
    raise FileNotFoundError(
        '공식 저장소에는 WPT-XLSR-AASIST 체크포인트가 없습니다. '
        '검증된 anti-spoofing_feat_model.pt의 로컬 경로 또는 직접 URL을 설정하세요.'
    )

def directory_size(path):
    return sum(p.stat().st_size for p in Path(path).rglob('*') if p.is_file())

print('도우미 함수 준비 완료')


도우미 함수 준비 완료


In [14]:
# 1) Baseline과 체크포인트 확인
from pathlib import Path
import os
import gc
import shutil
import time

WORK_DIR = WORK_DIR.resolve()

print("현재 작업 경로:", Path.cwd())
print("삭제 대상 경로:", WORK_DIR)

# 현재 작업 경로가 WORK_DIR 또는 그 하위라면 폴더 밖으로 이동
current_dir = Path.cwd().resolve()

if current_dir == WORK_DIR or WORK_DIR in current_dir.parents:
    os.chdir(WORK_DIR.parent)
    print("작업 경로 변경:", Path.cwd())

# Python이 가지고 있는 불필요한 객체 참조 정리
gc.collect()

# Windows에서 파일 핸들이 늦게 해제되는 경우를 고려해 재시도
if WORK_DIR.exists():
    for attempt in range(3):
        try:
            shutil.rmtree(WORK_DIR)
            break
        except PermissionError:
            if attempt == 2:
                raise
            gc.collect()
            time.sleep(0.5)

WORK_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("작업 폴더 준비 완료:", WORK_DIR)

CHECKPOINT = resolve_checkpoint(WORK_DIR / 'downloads')
if CHECKPOINT.stat().st_size < 1_000_000:
    raise ValueError(f'체크포인트가 비정상적으로 작습니다: {CHECKPOINT.stat().st_size:,} bytes')
OPEN_EXTRACT = WORK_DIR / 'open_extract'
BASELINE_ZIP, DATA_ROOT = find_baseline_submit(OPEN_OR_BASELINE_ZIP, OPEN_EXTRACT)

print('Baseline:', BASELINE_ZIP)
print('WPT checkpoint:', CHECKPOINT, f'({CHECKPOINT.stat().st_size / 2**30:.2f} GiB)')
print('checkpoint SHA256:', sha256(CHECKPOINT))

현재 작업 경로: c:\Users\Admin\Desktop\Dacon-Contest-DL
삭제 대상 경로: C:\Users\Admin\Desktop\Dacon-Contest-DL\wpt_submit_work_2
작업 폴더 준비 완료: C:\Users\Admin\Desktop\Dacon-Contest-DL\wpt_submit_work_2
Baseline: C:\Users\Admin\Desktop\Dacon-Contest-DL\wpt_submit_work_2\open_extract\baseline_submit.zip
WPT checkpoint: C:\Users\Admin\Desktop\Dacon-Contest-DL\WPT_XLSR_AASIST_FoR_20000\best.pt (1.18 GiB)
checkpoint SHA256: ab1ead8d47c707e1f750dff4f1bfe4a7aeeb9cefdc0815fbd215ea7d99dfc9a6


In [15]:
# 2) 공식 WPT 코드와 XLS-R 300M을 로컬에 준비
REPO_DIR = WORK_DIR / 'All-Type-ADD'
run(['git', 'clone', WPT_REPO, str(REPO_DIR)])
run(['git', 'checkout', WPT_COMMIT], cwd=REPO_DIR)
actual_commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
assert actual_commit == WPT_COMMIT

try:
    from huggingface_hub import snapshot_download
except ImportError:
    run([sys.executable, '-m', 'pip', 'install', 'huggingface-hub==0.34.4'])
    from huggingface_hub import snapshot_download

XLSR_DIR = WORK_DIR / 'xlsr_300m'
snapshot_download(
    repo_id=XLSR_REPO,
    local_dir=XLSR_DIR,
    ignore_patterns=['*.h5', '*.msgpack', '*.onnx', '*.ot'],
)
if not (XLSR_DIR / 'config.json').is_file():
    raise FileNotFoundError('XLS-R config.json 다운로드 실패')
if not any((XLSR_DIR / name).is_file() for name in ('model.safetensors', 'pytorch_model.bin')):
    raise FileNotFoundError('XLS-R 모델 가중치 다운로드 실패')
print('공식 코드 commit:', actual_commit)
print('XLS-R 크기:', f'{directory_size(XLSR_DIR) / 2**30:.2f} GiB')


+ git clone https://github.com/xieyuankun/All-Type-ADD.git C:\Users\Admin\Desktop\Dacon-Contest-DL\wpt_submit_work_2\All-Type-ADD
+ git checkout 4faae8ba700aa93f03cfc87c100dc626b6a7e68b


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

공식 코드 commit: 4faae8ba700aa93f03cfc87c100dc626b6a7e68b
XLS-R 크기: 1.18 GiB


In [16]:
# 3) Baseline 제출물을 풀고 DF-Arena를 WPT-XLSR-AASIST 자산으로 교체
STAGE = WORK_DIR / 'submit'
safe_extract(BASELINE_ZIP, STAGE)
for required in ('model', 'script.py', 'requirements.txt'):
    if not (STAGE / required).exists():
        raise FileNotFoundError(f'Baseline에 {required}가 없습니다')

df_arena = STAGE / 'model' / 'df_arena_1b'
if df_arena.exists():
    shutil.rmtree(df_arena)

WPT_MODEL_DIR = STAGE / 'model' / 'wpt_xlsr_aasist'
WPT_CODE_DIR = WPT_MODEL_DIR / 'code'
WPT_XLSR_DIR = WPT_MODEL_DIR / 'xlsr_300m'
WPT_CODE_DIR.mkdir(parents=True, exist_ok=True)
shutil.copytree(XLSR_DIR, WPT_XLSR_DIR, dirs_exist_ok=True)
shutil.copy2(CHECKPOINT, WPT_MODEL_DIR / 'anti-spoofing_feat_model.pt')

for name in ('model.py', 'feature_extraction.py'):
    shutil.copy2(REPO_DIR / name, WPT_CODE_DIR / name)
(WPT_CODE_DIR / 'exp').mkdir(exist_ok=True)
shutil.copy2(REPO_DIR / 'exp' / 'feature_extraction_exp.py', WPT_CODE_DIR / 'exp' / 'feature_extraction_exp.py')
(WPT_CODE_DIR / 'exp' / '__init__.py').write_text('', encoding='utf-8')

# 사용하지 않는 선택 의존성을 제거하고, 추론 로그/attention 출력을 끕니다.
model_py = (WPT_CODE_DIR / 'model.py').read_text(encoding='utf-8')
model_py = model_py.replace('from pytorch_model_summary import summary\n', '')
model_py = model_py.replace('import torchvision.models as models\n', '')
(WPT_CODE_DIR / 'model.py').write_text(model_py, encoding='utf-8')

wpt_features = (WPT_CODE_DIR / 'exp' / 'feature_extraction_exp.py').read_text(encoding='utf-8')
wpt_features = wpt_features.replace('self.model.config.output_attentions = True', 'self.model.config.output_attentions = False')
wpt_features = wpt_features.replace("            print(hidden_state.shape,'hidden_state')\n", '')
(WPT_CODE_DIR / 'exp' / 'feature_extraction_exp.py').write_text(wpt_features, encoding='utf-8')
print('모델 자산 교체 완료')


모델 자산 교체 완료


In [17]:
# 4) Baseline script.py에서 DF-Arena 추론부만 교체
script_path = STAGE / 'script.py'
script = script_path.read_text(encoding='utf-8')

old_path = 'DF_ARENA_DIR = MODEL_DIR / "df_arena_1b"'
new_path = '''WPT_DIR = MODEL_DIR / "wpt_xlsr_aasist"
WPT_CODE_DIR = WPT_DIR / "code"
WPT_XLSR_DIR = WPT_DIR / "xlsr_300m"
WPT_CHECKPOINT = WPT_DIR / "anti-spoofing_feat_model.pt"'''
if old_path not in script:
    raise RuntimeError('Baseline 경로 블록을 찾지 못했습니다')
script = script.replace(old_path, new_path, 1)

wpt_section = r'''# -----------------------------------------------------------------------------
# 5. WPT-XLSR-AASIST를 이용한 성분별 Fake 추론
# -----------------------------------------------------------------------------

def load_wpt_model(device):
    if str(WPT_CODE_DIR) not in sys.path:
        sys.path.insert(0, str(WPT_CODE_DIR))
    from model import WPTW2V2AASIST

    model = WPTW2V2AASIST(
        model_dir=str(WPT_XLSR_DIR),
        prompt_dim=1024,
        device=device.type,
        sampling_rate=AUDIO_SAMPLE_RATE,
        num_prompt_tokens=6,
        num_wavelet_tokens=4,
        dropout=0.0,
        visual=False,
    )
    checkpoint = torch.load(
        WPT_CHECKPOINT, map_location="cpu", weights_only=True
    )
    for key in ("state_dict", "model_state_dict", "model"):
        if isinstance(checkpoint, dict) and key in checkpoint and isinstance(checkpoint[key], dict):
            checkpoint = checkpoint[key]
            break
    if not isinstance(checkpoint, dict):
        raise TypeError("WPT checkpoint must contain a state_dict")
    if checkpoint and all(str(key).startswith("module.") for key in checkpoint):
        checkpoint = {str(key)[7:]: value for key, value in checkpoint.items()}

    incompatible = model.load_state_dict(checkpoint, strict=False)
    if incompatible.missing_keys or incompatible.unexpected_keys:
        raise RuntimeError(
            f"WPT checkpoint mismatch. Missing={incompatible.missing_keys[:10]}, "
            f"Unexpected={incompatible.unexpected_keys[:10]}"
        )
    model = model.to(device)
    model.eval()  # 공식 클래스의 eval()은 self를 반환하지 않으므로 별도 호출
    return model


def calculate_rms(audio):
    return float(np.sqrt(np.mean(np.square(audio, dtype=np.float64))))


def predict_fake(model, audio):
    if calculate_rms(audio) < SILENCE_RMS:
        return 0.0

    segment_scores = []
    for start in get_segment_starts(audio.size):
        segment = extract_segment(audio, start)
        # 공식 WPT 전처리기가 내부에서 device로 옮기므로 CPU [B, T] 텐서를 전달한다.
        segment_tensor = torch.from_numpy(segment).unsqueeze(0)
        with torch.inference_mode():
            _, logits = model(segment_tensor)
            probabilities = torch.softmax(logits.float(), dim=-1)
        # 공식 학습/evaluation 코드에서 class 0이 FAKE 점수이다.
        segment_scores.append(float(probabilities[0, 0]))

    return max(segment_scores)


'''
section_pattern = re.compile(
    r'# -+\n# 5\. DF-Arena 1B.*?(?=# -+\n# 6\.)',
    flags=re.DOTALL,
)
script, count = section_pattern.subn(wpt_section, script, count=1)
if count != 1:
    raise RuntimeError(f'DF-Arena section replacement count={count}')

replacements = {
    'df_arena_model, fake_label_index = load_df_arena_model(device)':
        'wpt_model = load_wpt_model(device)',
    'voice_fake = predict_fake(\n            df_arena_model, fake_label_index, voice_audio, device\n        )':
        'voice_fake = predict_fake(wpt_model, voice_audio)',
    'music_fake = predict_fake(\n            df_arena_model, fake_label_index, music_audio, device\n        )':
        'music_fake = predict_fake(wpt_model, music_audio)',
}
for old, new in replacements.items():
    if old not in script:
        raise RuntimeError(f'Baseline 호출부를 찾지 못했습니다: {old[:60]}')
    script = script.replace(old, new, 1)

script = script.replace('DF-Arena 1B', 'WPT-XLSR-AASIST')
if 'df_arena' in script.lower():
    raise RuntimeError('DF-Arena 참조가 script.py에 남아 있습니다')
script_path.write_text(script, encoding='utf-8')

requirements = (STAGE / 'requirements.txt').read_text(encoding='utf-8').rstrip()
if 'pytorch-wavelets' not in requirements.lower():
    requirements += '\npytorch-wavelets==1.3.0\n'
(STAGE / 'requirements.txt').write_text(requirements.lstrip(), encoding='utf-8')
print('script.py 교체 완료')


script.py 교체 완료


In [18]:
# 5) 출처·무결성 기록, 문법 및 zip 구조 검증
manifest = {
    'architecture': 'PANNs + HTDemucs + WPT-XLSR-AASIST',
    'wpt_repository': WPT_REPO,
    'wpt_commit': WPT_COMMIT,
    'wpt_checkpoint_sha256': sha256(WPT_MODEL_DIR / 'anti-spoofing_feat_model.pt'),
    'xlsr_repository': XLSR_REPO,
    'xlsr_license': 'Apache-2.0',
    'note': 'Verify the WPT code/checkpoint license and provenance before competition use.',
}
(WPT_MODEL_DIR / 'SOURCE_MANIFEST.json').write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8'
)

for py_file in (STAGE / 'script.py', WPT_CODE_DIR / 'model.py', WPT_CODE_DIR / 'feature_extraction.py', WPT_CODE_DIR / 'exp' / 'feature_extraction_exp.py'):
    source = py_file.read_text(encoding='utf-8')
    compile(source, str(py_file), 'exec')
    print('syntax OK:', py_file.relative_to(STAGE))

if SUBMIT_ZIP.exists():
    SUBMIT_ZIP.unlink()
with zipfile.ZipFile(SUBMIT_ZIP, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=1, allowZip64=True) as zf:
    for top_name in ('model', 'script.py', 'requirements.txt'):
        top_path = STAGE / top_name
        if top_path.is_dir():
            for path in sorted(top_path.rglob('*')):
                if path.is_file() and '__pycache__' not in path.parts:
                    zf.write(path, path.relative_to(STAGE).as_posix())
        else:
            zf.write(top_path, top_name)

with zipfile.ZipFile(SUBMIT_ZIP) as zf:
    names = zf.namelist()
    top_levels = {name.split('/', 1)[0] for name in names}
    if top_levels != {'model', 'script.py', 'requirements.txt'}:
        raise ValueError(f'잘못된 zip 최상위 구조: {top_levels}')
    uncompressed = sum(info.file_size for info in zf.infolist())
    bad = zf.testzip()
    if bad:
        raise ValueError(f'손상된 zip member: {bad}')

compressed = SUBMIT_ZIP.stat().st_size
if compressed > 10 * 2**30:
    raise ValueError(f'압축 파일이 10GB를 초과합니다: {compressed / 2**30:.2f} GiB')
if uncompressed > 32 * 2**30:
    raise ValueError(f'압축 해제 크기가 32GB를 초과합니다: {uncompressed / 2**30:.2f} GiB')

print('\n제출 파일 생성 완료:', SUBMIT_ZIP.resolve())
print(f'압축 크기: {compressed / 2**30:.2f} GiB')
print(f'해제 크기: {uncompressed / 2**30:.2f} GiB')
print('submit.zip SHA256:', sha256(SUBMIT_ZIP))


syntax OK: script.py
syntax OK: model\wpt_xlsr_aasist\code\model.py
syntax OK: model\wpt_xlsr_aasist\code\feature_extraction.py
syntax OK: model\wpt_xlsr_aasist\code\exp\feature_extraction_exp.py

제출 파일 생성 완료: C:\Users\Admin\Desktop\Dacon-Contest-DL\output_artifacts\submit.zip
압축 크기: 1.84 GiB
해제 크기: 2.74 GiB
submit.zip SHA256: 883c20f5af55cec55d8151d1949a37386183d6600e453fe8a04c136e57abe231


In [19]:
# 6) 선택 사항: Dacon open.zip의 더미 3개 파일로 end-to-end smoke test
# 실제 WPT 추론은 CUDA를 권장하며, 이 테스트는 시간이 걸릴 수 있습니다.
if RUN_LOCAL_SMOKE_TEST:
    if DATA_ROOT is None or not (DATA_ROOT / 'data' / 'test').is_dir():
        raise FileNotFoundError('open.zip에서 data/test를 찾을 수 없습니다')
    run([sys.executable, str(STAGE / 'script.py')], cwd=DATA_ROOT)
    submission = DATA_ROOT / 'output' / 'submission.csv'
    if not submission.is_file():
        raise FileNotFoundError('smoke test가 output/submission.csv를 만들지 못했습니다')
    import pandas as pd
    df = pd.read_csv(submission)
    expected = ['ID', 'FILE_FAKE_PROB', 'VOICE_FAKE_PROB', 'MUSIC_FAKE_PROB', 'VOICE_PRESENT_PROB', 'MUSIC_PRESENT_PROB']
    assert list(df.columns) == expected
    assert len(df) == 3
    assert df[expected[1:]].apply(lambda c: c.between(0, 1).all()).all()
    display(df)
else:
    print('Smoke test 생략. 필요하면 RUN_LOCAL_SMOKE_TEST=True로 바꾸고 다시 실행하세요.')


Smoke test 생략. 필요하면 RUN_LOCAL_SMOKE_TEST=True로 바꾸고 다시 실행하세요.


## 실행 순서

1. Dacon 데이터 탭에서 `open.zip`을 받고 `OPEN_OR_BASELINE_ZIP`을 맞춥니다.
2. 논문 저자 또는 정당한 배포처에서 받은 **co-trained WPT-XLSR-AASIST** 체크포인트를 `WPT_CHECKPOINT`에 지정합니다. 음성 전용 체크포인트보다 speech/sound/singing/music 공동학습 체크포인트가 본 대회의 음악 성분 탐지 목적에 더 잘 맞습니다.
3. 위에서 아래로 모든 셀을 실행합니다.
4. 마지막에 출력되는 `output_artifacts/submit.zip`을 Dacon 제출 탭에 업로드합니다.
5. 최초 실제 제출 전에는 가능하면 `RUN_LOCAL_SMOKE_TEST=True`로 더미 3개 파일 추론을 통과시키고, zip SHA256과 체크포인트 출처를 보관하세요.

### 확인해야 할 제한

- 공식 WPT 코드 저장소에는 명시적 `LICENSE` 파일이 없습니다. 대회 규칙의 '최소 비영리 사용 허용' 조건을 만족하는지 코드·체크포인트 권리자에게 확인해야 합니다.
- 실제 1,200개 파일에서 60분 제한을 만족하는지는 공개 더미 3개만으로 보장할 수 없습니다. 유효 체크포인트를 확보한 뒤 L4 또는 유사 GPU에서 충분한 길이의 오디오로 시간을 측정하세요.
- 체크포인트의 라벨 순서는 공식 평가 코드와 동일하게 **class 0 = FAKE**여야 합니다. 다른 포맷의 체크포인트라면 그대로 제출하지 마세요.
